# 001a — Prepare Train Data (Multimodal v2)

新增臨床特徵：`age`、`gender`

特徵向量（516 維）：
- `emb_0` ~ `emb_511`：HeAR embedding（512維）
- `age`：年齡（數值，標準化）
- `gender`：性別（male=0, female=1, other=2）
- `respiratory_condition`：慢性呼吸道疾病（0/1）
- `fever_muscle_pain`：發燒或肌肉痠痛（0/1）

## Imports

In [1]:
import os
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from collections import Counter
from tqdm.auto import tqdm
from sklearn.utils import resample

OUTPUT_DIR = 'data'
AUG_DIR    = 'data/augmented_wav'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(AUG_DIR, exist_ok=True)

c:\Users\aint\anaconda3\envs\base1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 設定路徑

In [2]:
COUGHVID_ROOT = 'C:/Users/aint/Desktop/RNN/Final_project/coughvid_wav'
METADATA_CSV  = 'C:/Users/aint/Desktop/RNN/Final_project/metadata_compiled.csv'
COUGH_DETECTED_THRESHOLD = 0.8

AUG_TARGET = {
    'obstructive_disease': 500,
    'COVID-19':            700,
    'upper_infection':     None,
    'lower_infection':     None,
    'healthy_cough':       None,
}

## 讀取 metadata & 前處理臨床特徵

In [3]:
df_meta = pd.read_csv(METADATA_CSV)
print('原始樣本數:', len(df_meta))

df_meta = df_meta[df_meta['cough_detected'] > COUGH_DETECTED_THRESHOLD].copy()
print('cough_detected 過濾後:', len(df_meta))

# respiratory_condition & fever_muscle_pain：NaN 補 False
df_meta['respiratory_condition'] = df_meta['respiratory_condition'].fillna(False).astype(int)
df_meta['fever_muscle_pain']     = df_meta['fever_muscle_pain'].fillna(False).astype(int)

# age：NaN 補中位數
age_median = df_meta['age'].median()
df_meta['age'] = df_meta['age'].fillna(age_median).astype(float)
print(f'age 中位數（補 NaN 用）: {age_median}')
print(f'age NaN 數量: {df_meta["age"].isna().sum()}')

# gender：male=0, female=1, other=2，NaN 補 -1（unknown）
gender_map = {'male': 0, 'female': 1, 'other': 2}
df_meta['gender_encoded'] = df_meta['gender'].map(gender_map).fillna(-1).astype(int)
print('\ngender 分佈:')
print(df_meta['gender'].value_counts(dropna=False))

原始樣本數: 34434
cough_detected 過濾後: 18541
age 中位數（補 NaN 用）: 35.0
age NaN 數量: 0

gender 分佈:
gender
male      8638
NaN       5008
female    4833
other       62
Name: count, dtype: int64


## 多數決

In [4]:
VALID_DIAGNOSES = {'COVID-19', 'healthy_cough', 'upper_infection', 'lower_infection', 'obstructive_disease'}
EXPERT_COLS     = ['diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_4']

def majority_vote(row):
    votes = [row[col] for col in EXPERT_COLS if pd.notna(row.get(col)) and row[col] in VALID_DIAGNOSES]
    if not votes:
        return None
    count = Counter(votes)
    top = count.most_common()
    if len(top) > 1 and top[0][1] == top[1][1]:
        return None
    return top[0][0]

df_meta['expert_label_voted'] = df_meta.apply(majority_vote, axis=1)
df_expert = df_meta[df_meta['expert_label_voted'].isin(VALID_DIAGNOSES)].copy()
df_expert = df_expert.rename(columns={'expert_label_voted': 'expert_label'})

print('有效專家標註樣本數:', len(df_expert))
print(df_expert['expert_label'].value_counts())

有效專家標註樣本數: 2416
expert_label
upper_infection        681
lower_infection        561
healthy_cough          543
COVID-19               470
obstructive_disease    161
Name: count, dtype: int64


## 五類 → 三分類

In [5]:
LABEL_MAP = {
    'COVID-19':            'covid',
    'healthy_cough':       'healthy',
    'upper_infection':     'symptomatic',
    'lower_infection':     'symptomatic',
    'obstructive_disease': 'symptomatic',
}
df_expert['label'] = df_expert['expert_label'].map(LABEL_MAP)
print('三分類分佈:')
print(df_expert['label'].value_counts())

三分類分佈:
label
symptomatic    1403
healthy         543
covid           470
Name: count, dtype: int64


## 確認 .wav 存在

In [6]:
def find_wav(uuid):
    path = os.path.join(COUGHVID_ROOT, f'{uuid}.wav')
    return path if os.path.exists(path) else None

df_expert['file_path'] = df_expert['uuid'].apply(find_wav)
df_expert = df_expert.dropna(subset=['file_path']).reset_index(drop=True)
print(f'最終可用樣本數: {len(df_expert)}')

最終可用樣本數: 2416


## 資料擴增

In [7]:
AUGMENT_METHODS = ['stretch_slow', 'stretch_fast', 'pitch_up', 'pitch_down', 'noise']

def augment_audio(audio, sr, method):
    if method == 'stretch_slow':
        return librosa.effects.time_stretch(audio, rate=0.9)
    elif method == 'stretch_fast':
        return librosa.effects.time_stretch(audio, rate=1.1)
    elif method == 'pitch_up':
        return librosa.effects.pitch_shift(audio, sr=sr, n_steps=1)
    elif method == 'pitch_down':
        return librosa.effects.pitch_shift(audio, sr=sr, n_steps=-1)
    elif method == 'noise':
        return audio + np.random.randn(len(audio)) * 0.005

def generate_augmented_samples(df_sub, expert_label, target_count):
    original_count = len(df_sub)
    needed = target_count - original_count
    if needed <= 0:
        print(f'{expert_label}: 已有 {original_count} 筆，不需擴增')
        return pd.DataFrame()

    print(f'{expert_label}: {original_count} → 目標 {target_count}（擴增 {needed} 筆）')
    method_cycle = (AUGMENT_METHODS * ((needed // len(AUGMENT_METHODS)) + 1))[:needed]
    sampled = df_sub.sample(needed, replace=True, random_state=42).reset_index(drop=True)

    aug_rows = []
    for i, row in sampled.iterrows():
        method = method_cycle[i]
        try:
            audio, sr  = librosa.load(row['file_path'], sr=None, mono=True)
            aug_audio   = augment_audio(audio, sr, method)
            aug_filename = f"aug_{row['uuid']}_{method}.wav"
            aug_path     = os.path.join(AUG_DIR, aug_filename)
            sf.write(aug_path, aug_audio, sr)

            new_row = row.copy()
            new_row['uuid']         = f"aug_{row['uuid']}_{method}"
            new_row['file_path']    = aug_path
            new_row['expert_label'] = expert_label
            new_row['label']        = LABEL_MAP[expert_label]
            # 臨床特徵繼承自原始樣本
            aug_rows.append(new_row)
        except Exception as e:
            print(f'  擴增失敗 {row["uuid"]} ({method}): {e}')

    return pd.DataFrame(aug_rows)

aug_dfs = []
for expert_label, target in AUG_TARGET.items():
    if target is None:
        continue
    df_sub = df_expert[df_expert['expert_label'] == expert_label]
    df_aug = generate_augmented_samples(df_sub, expert_label, target)
    if len(df_aug) > 0:
        aug_dfs.append(df_aug)

df_augmented = pd.concat([df_expert] + aug_dfs, ignore_index=True) if aug_dfs else df_expert.copy()
print('\n擴增後三分類分佈:')
print(df_augmented['label'].value_counts())

obstructive_disease: 161 → 目標 500（擴增 339 筆）
COVID-19: 470 → 目標 700（擴增 230 筆）

擴增後三分類分佈:
label
symptomatic    1742
covid           700
healthy         543
Name: count, dtype: int64


## Balancing

In [8]:
df_covid   = df_augmented[df_augmented['label'] == 'covid']
df_healthy = df_augmented[df_augmented['label'] == 'healthy']
df_symp    = df_augmented[df_augmented['label'] == 'symptomatic']

three_class_min = min(len(df_covid), len(df_healthy), len(df_symp))
print(f'Balancing 目標: {three_class_min}')

df_balanced = pd.concat([
    resample(df_covid,   replace=False, n_samples=three_class_min, random_state=42),
    resample(df_healthy, replace=False, n_samples=three_class_min, random_state=42),
    resample(df_symp,    replace=False, n_samples=three_class_min, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)

print('Balancing 後:')
print(df_balanced['label'].value_counts())
print('總樣本數:', len(df_balanced))

Balancing 目標: 543
Balancing 後:
label
healthy        543
covid          543
symptomatic    543
Name: count, dtype: int64
總樣本數: 1629


## 輸出清單（含四項臨床特徵）

In [9]:
out_cols = ['uuid', 'file_path', 'label', 'expert_label',
            'age', 'gender_encoded',
            'respiratory_condition', 'fever_muscle_pain']
df_list = df_balanced[out_cols].copy()
df_list['filename'] = df_list['file_path'].apply(os.path.basename)

out_path = os.path.join(OUTPUT_DIR, 'prepared_train_coughvid_list.csv')
df_list.to_csv(out_path, index=False)
print(f'已儲存至 {out_path}')
print(df_list[['label','age','gender_encoded','respiratory_condition','fever_muscle_pain']].describe())
df_list.head()

已儲存至 data\prepared_train_coughvid_list.csv
               age  gender_encoded  respiratory_condition  fever_muscle_pain
count  1629.000000     1629.000000            1629.000000        1629.000000
mean     34.570902        0.230203               0.232658           0.181093
std      12.710272        0.634140               0.422656           0.385213
min       1.000000       -1.000000               0.000000           0.000000
25%      25.000000        0.000000               0.000000           0.000000
50%      35.000000        0.000000               0.000000           0.000000
75%      41.000000        1.000000               0.000000           0.000000
max      97.000000        2.000000               1.000000           1.000000


,uuid,file_path,label,expert_label,age,gender_encoded,respiratory_condition,fever_muscle_pain,filename
0,e36c0dda-a5cb-4398-ae8f-235320a729e4,C:/Users/aint/Desktop/RNN/Final_project/coughv...,healthy,healthy_cough,35.0,-1,0,0,e36c0dda-a5cb-4398-ae8f-235320a729e4.wav
1,aug_f1b271ee-6902-4cff-b260-cfdf9bc3fa1f_pitch_up,data/augmented_wav\aug_f1b271ee-6902-4cff-b260...,covid,COVID-19,46.0,0,0,1,aug_f1b271ee-6902-4cff-b260-cfdf9bc3fa1f_pitch...
2,7eac138a-818b-49d3-8ff8-2facdb910b71,C:/Users/aint/Desktop/RNN/Final_project/coughv...,covid,COVID-19,35.0,1,1,1,7eac138a-818b-49d3-8ff8-2facdb910b71.wav
3,fbd52f87-029f-4bd9-86ed-4743018722e8,C:/Users/aint/Desktop/RNN/Final_project/coughv...,covid,COVID-19,40.0,1,0,0,fbd52f87-029f-4bd9-86ed-4743018722e8.wav
4,96678e97-6a39-4932-9c5d-13981167bbd8,C:/Users/aint/Desktop/RNN/Final_project/coughv...,covid,COVID-19,24.0,0,0,0,96678e97-6a39-4932-9c5d-13981167bbd8.wav
